# Clinical Trial Designer — GRPO Training (Kaggle)

**OpenEnv Hackathon | Meta PyTorch × Scaler School of Technology**

**Platform:** Kaggle (T4 ×2 or P100, 16 GB VRAM, 30 hrs/week)

> Set `DRY_RUN = True` in Cell 2 for pipeline validation without GPU.


In [ ]:
%%capture
# Kaggle already has torch, numpy, scipy, pandas, matplotlib
!pip install unsloth
!pip install trl>=0.29.0
!pip install requests datasets


## 2. Configuration and Environment Connection


In [ ]:
import requests, json, os, random, re, csv
from datetime import datetime, timezone

# === CONFIGURATION ===
ENV_URL = "https://roopalgn-openenv-clinical-trial.hf.space"
DRY_RUN = False          # Set True for pipeline validation
MODEL_SIZE = "1.5b"      # "1.5b", "3b", or "7b"
EPISODES = 20            # Training episodes
SEED = 42

MODEL_PRESETS = {
    "1.5b": {"model_name": "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit", "lora_r": 8, "seq_len": 2048, "grad_accum": 4},
    "3b":   {"model_name": "unsloth/Qwen2.5-3B-Instruct-bnb-4bit",   "lora_r": 16, "seq_len": 3072, "grad_accum": 4},
    "7b":   {"model_name": "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",   "lora_r": 16, "seq_len": 4096, "grad_accum": 8},
}
preset = MODEL_PRESETS[MODEL_SIZE]

def env_reset(seed=None):
    payload = {"seed": seed} if seed is not None else {}
    resp = requests.post(f"{ENV_URL}/reset", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

def env_step(action_type, parameters=None, justification="", confidence=0.5):
    payload = {"action_type": action_type, "parameters": parameters or {}, "justification": justification, "confidence": confidence}
    resp = requests.post(f"{ENV_URL}/step", json=payload, timeout=30)
    resp.raise_for_status()
    return resp.json()

def env_ping():
    return requests.get(f"{ENV_URL}/ping", timeout=10).json()

print("Ping:", env_ping())
obs = env_reset(seed=42)
print("Reset OK:", obs.get("scenario_description", "")[:80])


## 3. Dry-Run Validation (random policy, no model needed)


In [ ]:
def run_dry_run(n_episodes=2, max_steps=50, base_seed=42):
    os.makedirs("outputs/grpo", exist_ok=True)
    results = []
    with open("outputs/grpo/reward_log.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["episode","seed","total_reward","steps","terminal_outcome","timestamp"])
        for ep in range(n_episodes):
            s = base_seed + ep
            random.seed(s)
            obs = env_reset(seed=s)
            total, steps, done = 0.0, 0, False
            while not done and steps < max_steps:
                a = random.choice(obs.get("available_actions", ["synthesize_conclusion"]))
                r = env_step(a)
                rw = r.get("reward", {})
                total += sum(rw.values()) if isinstance(rw, dict) else float(rw)
                done = r.get("done", False)
                obs = r.get("observation", obs)
                steps += 1
            w.writerow([ep, s, f"{total:.6f}", steps, "success" if done else "timeout", datetime.now(timezone.utc).isoformat()])
            results.append(total)
            print(f"  Ep {ep+1}/{n_episodes} | reward={total:.4f} | steps={steps}")
    print(f"Mean reward: {sum(results)/len(results):.4f}")

if DRY_RUN:
    print("=== DRY RUN ===")
    run_dry_run()
    print("=== Done. Set DRY_RUN=False for real training. ===")


## 4. Load Model with Unsloth (4-bit for T4/P100)


In [ ]:
if not DRY_RUN:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=preset["model_name"], max_seq_length=preset["seq_len"],
        dtype=None, load_in_4bit=True)
    model = FastLanguageModel.get_peft_model(model,
        r=preset["lora_r"], lora_alpha=preset["lora_r"]*2, lora_dropout=0.05,
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
        bias="none", use_gradient_checkpointing="unsloth")
    model.print_trainable_parameters()


## 5. Reward Function


In [ ]:
SYSTEM_PROMPT = """You are a clinical trial designer. Design a complete Phase I/II clinical trial.
Available actions: run_dose_escalation, observe_safety_signal, estimate_effect_size,
set_primary_endpoint, set_sample_size, set_inclusion_criteria, set_exclusion_criteria,
set_dosing_schedule, set_control_arm, set_randomization_ratio, set_blinding,
submit_to_fda_review, request_protocol_amendment, run_interim_analysis,
modify_sample_size, add_biomarker_stratification, run_primary_analysis,
synthesize_conclusion, enroll_patients
Respond: {"action_type": "<action>", "parameters": {...}}"""

def parse_action(text):
    match = re.search(r'\{[^{}]*"action_type"[^{}]*\}', text)
    if match:
        try: return json.loads(match.group())
        except: pass
    return {"action_type": "synthesize_conclusion", "parameters": {}}

def run_episode(model_response):
    try:
        obs = env_reset(seed=random.randint(0, 10000))
        total, done, steps = 0.0, False, 0
        actions = [parse_action(l) for l in model_response.split("\n") if "action_type" in l]
        if not actions: actions = [parse_action(model_response)]
        for a in actions:
            if done or steps >= 50: break
            r = env_step(a["action_type"], a.get("parameters", {}))
            rw = r.get("reward", 0.0)
            total += sum(rw.values()) if isinstance(rw, dict) else float(rw)
            done = r.get("done", False); steps += 1
        return total
    except Exception as e: print(f"Error: {e}"); return -2.0

def reward_func(completions, **kwargs):
    return [run_episode(c[-1]["content"] if isinstance(c, list) and c else str(c)) for c in completions]


## 6. Training Dataset and GRPO


In [ ]:
if not DRY_RUN:
    from datasets import Dataset
    from trl import GRPOConfig, GRPOTrainer

    scenarios = ["solid_tumor_chemo (NSCLC)", "autoimmune_biologic (RA)",
                 "cns_depression (TRD)", "rare_disease_orphan (Morquio A)"]
    prompts = [
        {"prompt": [{"role":"system","content":SYSTEM_PROMPT},
                    {"role":"user","content":f"Scenario: {s}. Design a Phase I/II trial step by step."}]}
        for s in scenarios
    ] * max(1, EPISODES // 4)
    train_dataset = Dataset.from_list(prompts)

    training_args = GRPOConfig(
        output_dir="checkpoints/grpo", num_generations=8, max_completion_length=512,
        temperature=0.7, learning_rate=5e-6, num_train_epochs=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=preset["grad_accum"], max_steps=EPISODES,
        warmup_ratio=0.05, weight_decay=0.01, max_grad_norm=1.0,
        logging_steps=1, save_steps=max(1, EPISODES//4), report_to="none", bf16=True)

    trainer = GRPOTrainer(model=model, args=training_args, tokenizer=tokenizer,
                          train_dataset=train_dataset, reward_funcs=[reward_func])
    print(f"Training: {EPISODES} steps x 8 rollouts")
    trainer.train()
    print("Training complete!")


## 7. Save Checkpoint to HuggingFace Hub


In [ ]:
if not DRY_RUN:
    from kaggle_secrets import UserSecretsClient
    from huggingface_hub import login
    try: login(token=UserSecretsClient().get_secret("HF_TOKEN"))
    except:
        from huggingface_hub import notebook_login
        notebook_login()

    REPO_ID = "Roopalgn/clinical-trial-designer-grpo"
    model.save_pretrained("checkpoints/grpo/final")
    tokenizer.save_pretrained("checkpoints/grpo/final")
    model.push_to_hub(REPO_ID, commit_message=f"GRPO {MODEL_SIZE} from Kaggle")
    tokenizer.push_to_hub(REPO_ID)
    print(f"Uploaded: https://huggingface.co/{REPO_ID}")


## 8. Reward Curve Plot


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

if not DRY_RUN and "trainer" in dir() and hasattr(trainer, "state"):
    rewards = [l.get("reward") for l in trainer.state.log_history if "reward" in l]
    if rewards:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.scatter(range(len(rewards)), rewards, alpha=0.3, s=20)
        w = min(20, len(rewards) // 3 + 1)
        ax.plot(range(w-1, len(rewards)), np.convolve(rewards, np.ones(w)/w, "valid"),
                "r-", lw=2, label=f"Rolling avg (w={w})")
        z = np.polyfit(range(len(rewards)), rewards, 1)
        ax.plot(range(len(rewards)), np.poly1d(z)(range(len(rewards))),
                "--g", label=f"Trend (slope={z[0]:.3f})")
        ax.set(xlabel="Episode", ylabel="Reward",
               title=f"Reward Curve - {MODEL_SIZE} on Kaggle")
        ax.legend(); ax.grid(alpha=0.3)
        os.makedirs("results", exist_ok=True)
        plt.savefig("results/reward_curve.png", dpi=150)
        plt.show()
